# 05 – Nevralt nettverk, evaluering og forklaring

**Mål:** trene et nettverk med ett skjult lag, forklare tensorformene og
undersøke feil på nye bilder.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn


In [ ]:
from pytorch_101.utils import CLASS_NAMES, download_fashion_mnist

train_data, test_data = download_fashion_mnist()
print(f"Treningsbilder: {len(train_data)}, testbilder: {len(test_data)}")


## Eksempel: et skjult lag og ReLU

I notatbok 04 brukte vi ett lineært lag fra 784 piksler til ti skårer.
Nå legger vi inn et skjult lag med ReLU før det siste lineære laget.
Treningsløkken fra 04 kan brukes også med denne modellen.

Et skjult lag lager mellomliggende representasjoner. ReLU erstatter negative
verdier med 0 og beholder positive. Flere lineære lag uten aktiveringsfunksjon mellom seg kan slås sammen til
ett lineært lag. Flere lag alene gjør derfor ikke modellen i stand til å
representere en ikke-lineær funksjon.

Her er et lite eksempel med tre innganger, fire skjulte nevroner og to utganger.

*Merk*: Det siste laget gir rå skårer, og vi bruker ingen aktiveringsfunksjon etter det.


In [ ]:
small_model = nn.Sequential(nn.Linear(3, 4), nn.ReLU(), nn.Linear(4, 2))
print(small_model(torch.ones(5, 3)).shape)
values = torch.tensor([-2.0, -1.0, 0.0, 1.0, 2.0])
print("Før ReLU:", values, "Etter:", nn.ReLU()(values))


### Nettverket som graf

![Et fullkoblet nettverk med tre innganger, fire skjulte nevroner med ReLU og to utganger.](../assets/05_nettverk.svg)

Figuren viser den lille modellen over, for **ett eksempel**. Alle innganger er
koblet til alle nevronene i neste lag. Hver forbindelse har en vekt.
Et skjult nevron beregner en vektet sum pluss et konstantledd, og bruker så ReLU.
Utgangene beregner vektede summer pluss konstantledd, uten ReLU.


### Nøtt: Hvorfor trenger nevrale nettverk aktiveringsfunksjoner?

Et nevralt nettverk består av flere lag som bearbeider informasjon.
Uten aktiveringsfunksjoner kan flere lineære lag slås sammen til ett lineært lag.
Men hva betyr det for hva nettverket kan lære?

Anta at vi ønsker å trene et nettverk til å tilnærme funksjonen $f(x) = x^2$
for $x$ mellom $-1$ og $1$.

![Til venstre: En rett linje treffer ikke alle tre markerte punktene på parabolen. Til høyre: Flere rette linjestykker følger parabolen tettere.](../assets/05_aktivering.svg)


1. **Lineære lag:** Hvorfor hjelper det ikke å legge til flere lineære lag
   uten aktiveringsfunksjoner når vi vil tilnærme $x^2$?

   Bruk to enkle lag, $h = ax + b$ og $y = ch + d$:
   Sett inn uttrykket for $h$ og vis at resultatet fortsatt er på formen
   $y = Ax + B$. Kan én rett linje treffe alle punktene $(-1, 1)$, $(0, 0)$ og $(1, 1)$?
2. **Aktiveringsfunksjoner:** ReLU er gitt ved $\mathrm{ReLU}(z) = \max(0, z)$.

   Hva skjer med grafen når vi bruker ReLU på et lineært uttrykk? 
   
   Forklar hvordan flere slike funksjoner kan kombineres for å lage en graf med flere rette linjestykker, som til sammen kan tilnærme en kurve.

<details>
<summary>Hint</summary>

Start med $y = c(ax + b) + d$ og gang ut parentesen.
Hva er stigningstallet, og hva er konstantleddet?

ReLU er null for negative innganger og lik inngangen for positive innganger. Hvordan kan flere slike funksjoner kombineres for å lage en graf med flere knekkpunkter?

</details>


**Mitt svar:**

1. Setter vi inn det første laget i det andre, får vi

   $$y = c(ax + b) + d = (ca)x + (cb + d) = Ax + B.$$

   Dermed er $A = ca$ og $B = cb + d$. Vi har fortsatt bare en rett linje.
   Hvert nytt lineære lag kan slås sammen med de foregående på samme måte.
   For lag med flere nevroner gjelder det samme med matriser i stedet for enkelttall.

   Punktene $(-1, 1)$ og $(1, 1)$ krever en vannrett linje, $y = 1$.
   Den treffer ikke $(0, 0)$. En rett linje kan derfor ikke gjengi $x^2$
   på hele intervallet, selv om den kan treffe noen punkter.

2. ReLU har en knekk der inngangen går fra negativ til positiv.
   Når skjulte nevroner beregner $\mathrm{ReLU}(ax + b)$, oppstår knekkpunktene der (ax+b=0). Ulike vekter og konstantledd kan plassere knekkpunktene på forskjellige steder.

Neste lineære lag kan vekte og summere disse bidragene. Resultatet blir en sammenhengende graf satt sammen av rette linjestykker, slik som i høyre del av figuren.

Med nok slike linjestykker kan nettverket komme svært nær $x^2$ på intervallet, selv om et endelig ReLU-nettverk ikke kan gjengi parabolen helt nøyaktig. Aktiveringsfunksjonen gjør dette mulig, mens treningen må finne vektene og konstantleddene som gir en god tilnærming.


## Oppgave 1: Lag Fashion-MNIST-nettverket

Skriv funksjonen `create_model(hidden_size)` som returnerer følgende modell (nettverk):
`Flatten → Linear(784, hidden_size) → ReLU → Linear(hidden_size, 10)`.

**Regn for hånd før du skriver kode.** Bruk 128 skjulte nevroner.
Hvor mange vekter og konstantledd har det første lineære laget?
Hvor mange har det siste? Har `Flatten` eller `ReLU` parametere som læres?
Skriv utregningen og totalen i svarcellen under. Bruk antall forbindelser
inn til hvert nevron som hjelp.

Opprett deretter `model = create_model(128)`. Vis formen etter hvert lag for en gruppe
på 32 nullbilder. Tell parametrene med kode og sammenlign med håndregningen.

<details>
<summary>Hint til koden</summary>

Bruk en vanlig funksjon som returnerer `nn.Sequential`. Iterer over lagene for å vise mellomformene.
Tell parametrene med `sum(parameter.numel() for parameter in model.parameters())`.

</details>


**Mitt svar:**

Første lineære lag: `784 * 128 = 100 352` vekter og `128` konstantledd,
til sammen `100 480` parametere.
Siste lineære lag: `128 * 10 = 1 280` vekter og `10` konstantledd,
til sammen `1 290` parametere.
`Flatten` og `ReLU` har ingen parametere som læres.
Totalt: `100 480 + 1 290 = 101 770` parametere.


In [ ]:
def create_model(hidden_size) -> nn.Sequential:
    return nn.Sequential(
        nn.Flatten(),
        nn.Linear(784, hidden_size),
        nn.ReLU(),
        nn.Linear(hidden_size, 10),
    )


model = create_model(128)
values = torch.zeros(32, 1, 28, 28)
print("Inndata:", values.shape)
for layer in model:
    values = layer(values)
    print(type(layer).__name__, values.shape)
print("Parametere:", sum(parameter.numel() for parameter in model.parameters()))


**Kontroll:** Kjør cellen for å sjekke løsningen.


In [ ]:
assert model(torch.zeros(32, 1, 28, 28)).shape == (32, 10), "Modellen skal gi formen (32, 10) for 32 bilder. Sjekk lagene og antall utverdier."

assert sum(p.numel() for p in model.parameters()) == 101770, "Nettverket skal ha 101770 parametere. Sjekk størrelsene på de to Linear-lagene."
assert model(torch.zeros(1, 1, 28, 28)).shape == (1, 10), "Modellen skal også fungere for ett bilde og gi formen (1, 10)."

print("Kontroll bestått!")


### Refleksjon

Skriv formene gjennom modellen med 128 skjulte nevroner.
Hva tilfører ReLU? Hvorfor har siste lag ti utganger, uansett størrelsen på det skjulte laget?

Skriv svaret ditt i Markdown-cellen under. Bruk egne resultater der det er relevant.


**Mitt svar:**

`[32, 1, 28, 28] → [32, 784] → [32, 128] → [32, 128] → [32, 10]`.
ReLU tilfører ikke-linearitet uten å endre formen. Den lar modellen lære
andre sammenhenger enn en samlet lineær transformasjon. De ti utgangene
tilsvarer de ti mulige klasserne, ikke antall skjulte nevroner.


## Oppgave 2: Lag en funksjon for trening

Gjør treningskoden din fra notatbok 04 om til funksjonen
`train_model(model, train_loader, num_epochs=25, learning_rate=0.001)`.

Funksjonen skal opprette kryssentropitap og en ny Adam-optimizer, trene modellen
og returnere en liste med gjennomsnittlig treningstap per epoke. Antall
treningsbilder finnes med `len(train_loader.dataset)`.

Skriv ut tapet etter hver epoke.
Senere skal vi bruke denne funksjonen til å trene nettverket.

<details>
<summary>Hint</summary>

Flytt løkken inn i en funksjon. Opprett optimizer inne i funksjonen slik at den tilhører modellen som sendes inn.

</details>

In [ ]:
def train_model(model, train_loader, num_epochs=25, learning_rate=0.001):
    loss_fn = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    history = []
    for epoch in range(num_epochs):
        model.train()
        loss_sum = 0.0
        for images, labels in train_loader:
            optimizer.zero_grad()
            scores = model(images)
            loss = loss_fn(scores, labels)
            loss.backward()
            optimizer.step()
            loss_sum += loss.item() * len(images)
        history.append(loss_sum / len(train_loader.dataset))
        print(f"Epoke {epoch + 1}: treningstap {history[-1]:.4f}")
    return history


## Oppgave 3: Lag en funksjon for evaluering

Skriv `evaluate_model(model, data_loader)` ved å tilpasse testløkken fra
notatbok 04. Returner en `dict` med:

- `accuracy`: antall riktige delt på antall bilder.
- `predictions`: alle predikerte klasser, samlet i én tensor.
- `labels`: alle riktige klasser, samlet i samme rekkefølge.

Bruk `eval()` og `no_grad()`. Funksjonen skal ikke oppdatere modellen.

<details>
<summary>Hint</summary>

Samle tensorer i to lister og bruk `torch.cat` til slutt. Bruk `argmax(dim=1)` for prediksjoner.

</details>


In [ ]:
def evaluate_model(model, data_loader):
    model.eval()
    all_predictions = []
    all_labels = []
    with torch.no_grad():
        for images, labels in data_loader:
            scores = model(images)
            all_predictions.append(scores.argmax(dim=1))
            all_labels.append(labels)
    predictions = torch.cat(all_predictions)
    labels = torch.cat(all_labels)
    accuracy = (predictions == labels).float().mean().item()
    return {"accuracy": accuracy, "predictions": predictions, "labels": labels}


## Innstillinger og tidtaking

Vi bruker **128 skjulte nevroner**, batchstørrelse 64, 25 epoker og Adam med læringsrate 0.001.
Hold disse innstillingene faste gjennom hovedløpet. Startvekter og rekkefølgen
på treningsbildene er tilfeldige, så resultatene kan variere mellom kjøringer.


In [ ]:
from time import perf_counter

from torch.utils.data import DataLoader

# Eksempel på tidtaking; bruk samme mønster rundt selve treningen.
start = perf_counter()
sum(range(10_000))
print(f"Eksempelberegningen tok {perf_counter() - start:.6f} sekunder")


## Oppgave 4: Tren nettverket og mål resultatene

Tren nettverket med **128 skjulte nevroner** og undersøk resultatet.
Bruk `create_model`, `train_model` og `evaluate_model` fra oppgave 1–3.

1. Opprett en ny modell og kall den `model`.
2. Opprett `train_loader` og `test_loader` med batchstørrelse 64. Stokk bare treningen.
3. Tren med `train_model` og lagre tapshistorikken i `history`.
   Mål tiden rundt selve treningen og lagre antall sekunder i `training_time`.
4. Evaluer med `evaluate_model` og lagre resultatet i `results`.
   Skriv ut sluttap, nøyaktighet og treningstid. Tegn treningskurven med aksetekster.

<details><summary>Hint</summary>

Bruk `perf_counter()` før og etter treningen.
`history` er en liste med tap per epoke. `results` er ordboken fra
`evaluate_model`, så testnøyaktigheten finnes i `results["accuracy"]`.

</details>


In [ ]:
model = create_model(128)
train_loader = DataLoader(
    train_data,
    batch_size=64,
    shuffle=True,
)
test_loader = DataLoader(test_data, batch_size=64, shuffle=False)
start = perf_counter()
history = train_model(model, train_loader)
training_time = perf_counter() - start
results = evaluate_model(model, test_loader)
print(f"Sluttap: {history[-1]:.4f}; testnøyaktighet: {results['accuracy']:.1%}; treningstid: {training_time:.2f} s")
fig, ax = plt.subplots()
ax.plot(range(1, 26), history, marker="o")
ax.set(xlabel="Epoke", ylabel="Gjennomsnittlig treningstap", title="Nettverk med 128 skjulte nevroner")
plt.show()


**Kontroll:** Kjør cellen for å sjekke løsningen.


In [ ]:
assert len(history) == 25, "history skal inneholde ett tap per epoke, totalt 25."
assert np.isfinite(history).all(), "Alle tap i history skal være endelige tall, uten nan eller inf."
assert history[-1] < history[0], "Siste treningstap skal være lavere enn det første. Sjekk treningsløkken og parameteroppdateringen."
assert results["predictions"].shape == (1000,), 'results["predictions"] skal inneholde én prediksjon per testbilde, med formen (1000,).'
assert 0 <= results["accuracy"] <= 1, 'results["accuracy"] skal være en andel mellom 0 og 1.'

print("Kontroll bestått!")


### Refleksjon

Oppgi første og siste treningstap, testnøyaktighet og tid fra din kjøring.
Hva kan vi konkludere med? Kan vi avgjøre om modellen overtilpasser bare ved å
se at treningstapet synker? Hvorfor bør vi ikke velge læringsrate ut fra testresultatet?

Sammenlign også testnøyaktigheten med den lineære modellen fra notatbok 04.
Hva observerer du? Er ett forsøk nok til å si at et nevralt nettverk alltid er bedre?


**Mitt svar:**

Koden over skriver ut mine målte tall. Fallende treningstap viser at modellen
tilpasser seg treningsdata etter tapsfunksjonens mål. Testnøyaktighet måler
riktige klassifikasjoner på det separate utvalget. Treningstap alene kan ikke
avsløre overtilpasning; under utvikling ville vi sammenlignet med et valideringssett.
Hvis vi velger læringsrate ut fra testresultater, påvirker testdata modellvalget
og blir mindre egnet til en uavhengig sluttevaluering.

Resultatet fra notatbok 04 gir en referanse for en modell uten skjult lag.
En eventuell forskjell gjelder disse kjøringene på de samme datautvalgene;
begge modellene bruker Adam, læringsrate 0.001, batchstørrelse 64 og 25 epoker.
Modellene har ulik arkitektur og forskjellige startvekter. Ett forsøk viser ikke at
et nevralt nettverk alltid er bedre enn en lineær modell.


## Eksempel: finn posisjoner som oppfyller en betingelse

`torch.where` kan finne posisjonene der to tensorer er forskjellige.
Dette bruker vi til å finne feilklassifiserte bilder.


In [ ]:
example_predictions = torch.tensor([1, 2, 3, 4])
example_targets = torch.tensor([1, 7, 3, 9])
error_positions = torch.where(example_predictions != example_targets)[0]
print("Feil på posisjonene:", error_positions.tolist())


## Oppgave 5: Vis modellens feil

Bruk den forhåndsvalgte modellen med **128 skjulte nevroner**.
Finn feilindeksene fra `results` i oppgave 4 og vis opptil ni feilklassifiserte
testbilder. Skriv riktig og predikert klassenavn i tittelen for hvert bilde.
Bruk `CLASS_NAMES` til å slå opp navnene fra klasseindeksene.
Håndter også at modellen kan ha færre enn ni feil: skjul ubrukte akser.

`test_loader` stokker ikke, så resultatene har samme rekkefølge som `test_data`.

<details>
<summary>Hint</summary>

Kombiner `torch.where` med rutenettet fra notatbok 02. Bruk `index.item()` når du indekserer datasettet.

</details>


In [ ]:
predictions = results["predictions"]
labels = results["labels"]
error_indices = torch.where(predictions != labels)[0][:9]
fig, axes = plt.subplots(3, 3, figsize=(10, 9))
for ax in axes.ravel():
    ax.axis("off")
for ax, index in zip(axes.ravel(), error_indices):
    position = index.item()
    image, label = test_data[position]
    ax.imshow(image.squeeze(0), cmap="gray", vmin=0, vmax=1)
    ax.set_title(f"Riktig: {CLASS_NAMES[label]}\nGjettet: {CLASS_NAMES[predictions[position].item()]}")
fig.suptitle("Eksempler på feilklassifiseringer")
fig.tight_layout()
plt.show()
print("Antall feil i hele testutvalget:", (predictions != labels).sum().item())


### Refleksjon

Beskriv to konkrete feil fra figuren. Ser noen av klassene like ut?
Er ni utvalgte feil nok til å si hvilken klasse modellen er dårligst på?
Hva måtte vi undersøkt for å svare bedre på det?

Skriv svaret ditt i Markdown-cellen under. Bruk egne resultater der det er relevant.


**Mitt svar:**

Et mulig eksempel er at en skjorte tolkes som en T-skjorte/topp fordi ermene
og omrisset ligner, eller at en kåpe tolkes som en genser. Beskriv to feil
fra din egen figur; hvilke feil som vises, kan variere. Visuell likhet er
en mulig forklaring, ikke et bevis på hvilke trekk modellen brukte.
De ni første feilene viser ikke hvilken klasse som er vanskeligst:
vi må telle riktige og feil per klasse i hele utvalget.


## Oppgave 6: Lag en forvekslingsmatrise

Lag en forvekslingsmatrise for den ferdigtrente 128-modellen: rader er riktige
klasser, kolonner er predikerte klasser. 

Finn selv ut hvordan du lager matrisen, f.eks. med [scikit-learns](https://scikit-learn.org/stable/modules/model_evaluation.html#confusion-matrix).
Du kan også skrive funksjonaliteten selv!

Visualiser matrisen med fargeskala, norske aksetekster og klassenavn langs begge
aksene. Undersøk hvilke forvekslinger som er vanligst. Bruk prediksjonene fra oppgavene over.


In [ ]:
matrix = torch.zeros(10, 10, dtype=torch.int64)
for target, predicted in zip(results["labels"], results["predictions"]):
    matrix[target, predicted] += 1
fig, ax = plt.subplots(figsize=(10, 8))
display = ax.imshow(matrix.numpy(), cmap="Blues")
fig.colorbar(display, ax=ax, label="Antall bilder")
ax.set(xlabel="Predikert klasse", ylabel="Riktig klasse", title="Forvekslingsmatrise")
ax.set_xticks(range(10), CLASS_NAMES, rotation=45, ha="right")
ax.set_yticks(range(10), CLASS_NAMES)
fig.tight_layout()
plt.show()


## Bonus: 8 eller 128 skjulte nevroner?

Tren et nytt nettverk med 8 skjulte nevroner og sammenlign med nettverket
med 128. Bruk samme batchstørrelse, læringsrate og antall epoker.
Sammenlign treningstap, testnøyaktighet og treningstid, og tegn
treningskurvene i samme figur.

En idé kan være å samle resultatene i ordbøker (`dict`), med nøklene `8` og `128`
for de to nettverkene. Du kan for eksempel bruke én ordbok for tapshistorikk,
én for treningstid og én for evalueringsresultater. Gjenbruk resultatene fra
128-modellen du allerede har trent.

Hva observerer du?


In [ ]:
histories = {128: history}
training_times = {128: training_time}
comparison_results = {128: results}

small_model = create_model(8)
small_loader = DataLoader(train_data, batch_size=64, shuffle=True)
start = perf_counter()
histories[8] = train_model(small_model, small_loader)
training_times[8] = perf_counter() - start
comparison_results[8] = evaluate_model(small_model, test_loader)
fig, ax = plt.subplots()
for size in [8, 128]:
    print(size, "sluttap:", histories[size][-1], "testnøyaktighet:", comparison_results[size]["accuracy"], "sekunder:", training_times[size])
    ax.plot(range(1, 26), histories[size], marker="o", label=f"{size} nevroner")
ax.set(xlabel="Epoke", ylabel="Gjennomsnittlig treningstap", title="To skjulte lagstørrelser")
ax.legend()
plt.show()


### Refleksjon

Oppgi begge modellresultatene fra din kjøring. Hvilken praktisk forskjell ser du?
Kan vi konkludere med at den ene størrelsen alltid er bedre?


**Mitt svar:**

Tabellen og figuren viser målt forskjell i tap, nøyaktighet og tid. Hvis
nøyaktigheten er lik, har den større modellen ikke gitt bedre resultat etter
dette målet i denne kjøringen. Én kjøring og 1 000 testbilder gir begrenset
støtte for generelle konklusjoner. Treningstid varierer også mellom maskiner.


## Referanser

- [3Blue1Brown – But what is a Neural Network?](https://www.3blue1brown.com/lessons/neural-networks/)
- [PyTorch – Build the Neural Network](https://docs.pytorch.org/tutorials/beginner/basics/buildmodel_tutorial.html)
- [Google Machine Learning Crash Course – Overfitting](https://developers.google.com/machine-learning/crash-course/overfitting/overfitting)